In [9]:
##### Converts raw rasters on crop and livestock production into final varaiables needed 
# pixel and subnational (vector) scale
# variables: 
    # % of production by crop/livestock grouping

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats
from collections import defaultdict

In [10]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# total production 
total_production = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# Save paths
vectors = f"{cd}/Data/Clean/Predictors/Vectors/commodity_mix.csv"

In [11]:
##### Define groupings

crop_groups = {
    # cereals
    'WHEA': 'cereals',
    'RICE': 'cereals',
    'BARL': 'cereals',
    'MAIZ': 'cereals',
    'OCER': 'cereals',
    'AGG_MILLET': 'cereals',
    'SORG': 'cereals',
    # fibres
    'COTT': 'fibres',
    'OFIB': 'fibres',
    # fruits
    'BANA': 'fruits',
    'PLNT': 'fruits',
    'CITR': 'fruits',
    'TEMF': 'fruits',
    'TROF': 'fruits',
    # oilcrops
    'SOYB': 'oilcrops',
    'GROU': 'oilcrops',
    'CNUT': 'oilcrops',
    'OILP': 'oilcrops',
    'OOIL': 'oilcrops',
    'SUNF': 'oilcrops',
    'RAPE': 'oilcrops',
    'SESA': 'oilcrops',
    # pulses
    'BEAN': 'pulses',
    'OPUL': 'pulses',
    'CHIC': 'pulses',
    'COWP': 'pulses',
    'PIGE': 'pulses',
    'LENT': 'pulses',
    # roots & tubers
    'POTA': 'roots_tubers',
    'SWPO': 'roots_tubers',
    'CASS': 'roots_tubers',
    'ORTS': 'roots_tubers',
    'YAMS': 'roots_tubers',
    # rest of crops
    'REST': 'rest_of_crops',
    'AGG_COFFEE': 'rest_of_crops',
    'COCO': 'rest_of_crops',
    'TEAS': 'rest_of_crops',
    'TOBA': 'rest_of_crops',
    # sugar crops
    'SUGC': 'sugar_crops',
    'SUGB': 'sugar_crops',
    # vegetables
    'VEGE': 'vegetables',
    'TOMA': 'vegetables',
    'ONIO': 'vegetables',
    # rubber
    'RUBB': 'rubber',
}

livestock_groups = {
    # ruminants
    'cattle': 'ruminants',
    'buffalo': 'ruminants',
    'sheep': 'ruminants',
    'goats': 'ruminants',
    # monogastrics
    'horses': 'monogastrics',
    'pigs': 'monogastrics',
    # poultry
    'ducks': 'poultry',
    'chicken': 'poultry',
    # other
    'all': 'other'
}

In [12]:
#### Run resampling for pixel scale 

### PREP 

# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_shape = ref.shape
    dst_meta  = ref.meta.copy()

# load total production 
with rasterio.open(total_production) as src:
    total = src.read(1).astype(np.float64)
    total[total == -9999] = np.nan

# invert groups to get {group: [crop1, crop2, ...]}
group_crops = defaultdict(list)
for crop, group in crop_groups.items():
    group_crops[group].append(crop)

group_livestock = defaultdict(list)
for animal, group in livestock_groups.items():
    group_livestock[group].append(animal)

### CALCULATE CROPS
# for each group: sum crops in group then divide by total
for group, crops in group_crops.items():

    group_sum = np.zeros(dst_shape, dtype=np.float64)

    for crop in crops:
        path = f"{cd}/Data/Clean/Production/Crop_value/{crop}_crop_value_USD_2020.tif"
        try:
            with rasterio.open(path) as src:
                data = src.read(1).astype(np.float64)
                data[data == -9999] = np.nan
                data[data == src.nodata] = np.nan if src.nodata is not None else data[data == src.nodata]
                group_sum = np.where(np.isnan(data), group_sum, group_sum + np.nan_to_num(data, nan=0))
        except Exception as e:
            print(f"WARNING: could not load {crop} — {e}")

    # divide by total to get share
    with np.errstate(invalid="ignore", divide="ignore"):
        share = np.where(
            (total > 0) & ~np.isnan(total),
            group_sum / total,
            np.nan
        )

    share = np.clip(share, 0, 1)

    # save
    out_meta = dst_meta.copy()
    out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

    out_arr = share.astype(np.float32)
    out_arr[np.isnan(out_arr)] = -9999

    out_path = f"{cd}/Data/Clean/Predictors/Rasters/{group}_share_production_USD.tif"
    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(out_arr, 1)

    print(f"Saved {group} ({len(crops)} crops) → {out_path}")


### CALCULATE LIVESTOCK
# for each group: sum livestock in group then divide by total
for group, animals in group_livestock.items():

    group_sum = np.zeros(dst_shape, dtype=np.float64)

    for animal in animals:
        path = f"{cd}/Data/Clean/Production/Livestock_value/{animal}_production_USD_2020.tif"
        try:
            with rasterio.open(path) as src:
                data = src.read(1).astype(np.float64)
                data[data == -9999] = np.nan
                if src.nodata is not None:
                    data[data == src.nodata] = np.nan
                group_sum = np.where(np.isnan(data), group_sum, group_sum + np.nan_to_num(data, nan=0))
        except Exception as e:
            print(f"WARNING: could not load {animal} — {e}")

    # share of total production
    with np.errstate(invalid="ignore", divide="ignore"):
        share = np.where(
            (total > 0) & ~np.isnan(total),
            group_sum / total,
            np.nan
        )

    share = np.clip(share, 0, 1)

    out_meta = dst_meta.copy()
    out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

    out_arr = share.astype(np.float32)
    out_arr[np.isnan(out_arr)] = -9999

    out_path = f"{cd}/Data/Clean/Predictors/Rasters/{group}_share_production_USD.tif"
    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(out_arr, 1)

    print(f"Saved {group} ({len(animals)} animals) → {out_path}")


Saved cereals (7 crops) → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/cereals_share_production_USD.tif
Saved fibres (2 crops) → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/fibres_share_production_USD.tif
Saved fruits (5 crops) → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/fruits_share_production_USD.tif
Saved oilcrops (8 crops) → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/oilcrops_share_production_USD.tif
Saved pulses (6 crops) → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/pulses_share_production_USD.tif
Saved roots_tubers (5 crops) → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/roots_tubers_share_production_USD.tif
Saved rest_of_crops (5 crops) → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/rest_of_crops_share_product

In [14]:
#### Run resampling for vector scale 
### TAKES 1 HOUR TO RUN ###

### PREP 
# reproject geo
gdf_proj  = total_geo.to_crs("EPSG:4326")

# prep results 
result = total_geo[["PROJ_ID"]].copy()


### CALCULATE
# step 1: sum total production per vector
zonal_total = rasterstats.zonal_stats(gdf_proj, total_production, stats="sum", nodata=-9999)
total_sums  = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_total])

# get crop shares
for group, crops in group_crops.items():

    group_sums = np.zeros(len(gdf_proj), dtype=np.float64)

    for crop in crops:
        path = f"{cd}/Data/Clean/Production/Crop_value/{crop}_crop_value_USD_2020.tif"
        try:
            zonal = rasterstats.zonal_stats(gdf_proj, path, stats="sum", nodata=-9999)
            vals  = np.array([z["sum"] if z["sum"] is not None else 0.0 for z in zonal])
            group_sums += np.nan_to_num(vals, nan=0)
        except Exception as e:
            print(f"WARNING: could not load {crop} — {e}")

    with np.errstate(invalid="ignore", divide="ignore"):
        share = np.where(
            (total_sums > 0) & ~np.isnan(total_sums),
            group_sums / total_sums,
            np.nan
        )

    result[f"{group}_share_production_USD"] = np.clip(share, 0, 1)
    print(f"Done: {group}")

# get livestock shares
for group, animals in group_livestock.items():

    group_sums = np.zeros(len(gdf_proj), dtype=np.float64)

    for animal in animals:
        path = f"/{cd}/Data/Clean/Production/Livestock_value/{animal}_production_USD_2020.tif"
        try:
            zonal = rasterstats.zonal_stats(gdf_proj, path, stats="sum", nodata=-9999)
            vals  = np.array([z["sum"] if z["sum"] is not None else 0.0 for z in zonal])
            group_sums += np.nan_to_num(vals, nan=0)
        except Exception as e:
            print(f"WARNING: could not load {animal} — {e}")

    with np.errstate(invalid="ignore", divide="ignore"):
        share = np.where(
            (total_sums > 0) & ~np.isnan(total_sums),
            group_sums / total_sums,
            np.nan
        )

    result[f"{group}_share_production_USD"] = np.clip(share, 0, 1)
    print(f"Done: {group}")

result.to_csv(vectors, index=False)


Done: cereals
Done: fibres
Done: fruits
Done: oilcrops
Done: pulses
Done: roots_tubers
Done: rest_of_crops
Done: sugar_crops
Done: vegetables
Done: rubber
Done: ruminants
Done: monogastrics
Done: poultry
Done: other
